# Varroa Mite Detection - Modular Notebook

This notebook is generated from the original all-in-one experiment notebook.

Outputs are intentionally cleared. API keys and personal secrets are not stored in this notebook.

## CNN Classification Experiments

Models: ResNet18 and EfficientNet-B0

Tasks:
- Dataset download and preparation
- YOLO-to-binary-classification CSV conversion
- Model training
- Internal validation
- External validation
- Result graphs


In [ ]:
!pip install -q torch torchvision torchaudio scikit-learn pandas matplotlib opencv-python pyyaml roboflow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path
import shutil
import yaml
from getpass import getpass

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from roboflow import Roboflow

PROJECT_DIR = '/content/drive/MyDrive/varroa_bitirme'
RAW_DIR = f'{PROJECT_DIR}/raw_datasets'
PROCESSED_DIR = f'{PROJECT_DIR}/processed_datasets'
CLASSIFICATION_DIR = f'{PROJECT_DIR}/classification_datasets'
CLASSIFICATION_RUNS_DIR = f'{PROJECT_DIR}/classification_runs'
GRAPH_DIR = f'{PROJECT_DIR}/github_cnn_yolo_outputs'

for d in [RAW_DIR, PROCESSED_DIR, CLASSIFICATION_DIR, CLASSIFICATION_RUNS_DIR, GRAPH_DIR]:
    os.makedirs(d, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Do not hard-code API keys in GitHub.
ROBOFLOW_API_KEY = getpass('Roboflow API Key: ')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

In [ ]:
def download_datasets():
    os.chdir(RAW_DIR)

    project_varroa = rf.workspace('varroa-j8231').project('varroa-bxxhd')
    dataset_varroa = project_varroa.version(1).download('yolov8')

    project_honeybee = rf.workspace('honeybee').project('honeybee_varroamite')
    dataset_honeybee = project_honeybee.version(5).download('yolov8')

    project_external = rf.workspace('varroa-virus-detection').project('varroa-mites-detector')
    dataset_external = project_external.version(2).download('yolov8')

    return dataset_varroa.location, dataset_honeybee.location, dataset_external.location

VARROA_RAW, HONEYBEE_RAW, EXTERNAL_RAW = download_datasets()

In [ ]:
def prepare_single_class_yolo_dataset(src_dir, dst_dir, source_varroa_class_ids, new_class_name='varroa', min_wh=1e-6, max_area=0.60):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)

    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    stats = {'copied_images':0,'total_label_lines':0,'kept_label_lines':0,'removed_wrong_class':0,'removed_invalid_bbox':0,'empty_labels_after_cleaning':0}

    for split in ['train','valid','test']:
        src_img_dir = src_dir / split / 'images'
        src_lbl_dir = src_dir / split / 'labels'
        dst_img_dir = dst_dir / split / 'images'
        dst_lbl_dir = dst_dir / split / 'labels'
        dst_img_dir.mkdir(parents=True, exist_ok=True)
        dst_lbl_dir.mkdir(parents=True, exist_ok=True)

        if not src_img_dir.exists():
            continue

        image_paths = []
        for ext in ['*.jpg','*.jpeg','*.png','*.bmp','*.webp']:
            image_paths.extend(src_img_dir.glob(ext))

        for img_path in image_paths:
            shutil.copy2(img_path, dst_img_dir / img_path.name)
            stats['copied_images'] += 1

            label_path = src_lbl_dir / f'{img_path.stem}.txt'
            new_lines = []
            if label_path.exists():
                lines = label_path.read_text().strip().splitlines()
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        stats['removed_invalid_bbox'] += 1
                        continue
                    stats['total_label_lines'] += 1
                    class_id = int(float(parts[0]))
                    x,y,w,h = map(float, parts[1:])

                    if class_id not in source_varroa_class_ids:
                        stats['removed_wrong_class'] += 1
                        continue
                    if w <= min_wh or h <= min_wh or (w*h) > max_area:
                        stats['removed_invalid_bbox'] += 1
                        continue

                    new_lines.append(f'0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
                    stats['kept_label_lines'] += 1

            if len(new_lines) == 0:
                stats['empty_labels_after_cleaning'] += 1
            (dst_lbl_dir / f'{img_path.stem}.txt').write_text('\n'.join(new_lines))

    data_yaml = {'path': str(dst_dir), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'names': {0: new_class_name}}
    with open(dst_dir / 'data.yaml', 'w') as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False)
    return stats

In [ ]:
VARROA_PROCESSED = f'{PROCESSED_DIR}/varroa_dataset_single_class'
HONEYBEE_PROCESSED = f'{PROCESSED_DIR}/honeybee_varroamite_single_class'
EXTERNAL_PROCESSED = f'{PROCESSED_DIR}/external_varroa_mites_detector_single_class'

prepare_single_class_yolo_dataset(VARROA_RAW, VARROA_PROCESSED, source_varroa_class_ids=[1], max_area=0.35)
prepare_single_class_yolo_dataset(HONEYBEE_RAW, HONEYBEE_PROCESSED, source_varroa_class_ids=[0], max_area=0.60)
prepare_single_class_yolo_dataset(EXTERNAL_RAW, EXTERNAL_PROCESSED, source_varroa_class_ids=[0], max_area=0.35)

In [ ]:
def create_binary_classification_csv(src_yolo_dir, output_csv_path):
    rows = []
    src_yolo_dir = Path(src_yolo_dir)

    for split in ['train', 'valid', 'test']:
        img_dir = src_yolo_dir / split / 'images'
        lbl_dir = src_yolo_dir / split / 'labels'
        if not img_dir.exists():
            continue

        image_paths = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']:
            image_paths.extend(img_dir.glob(ext))

        for img_path in image_paths:
            label_path = lbl_dir / f'{img_path.stem}.txt'
            class_name = 'no_varroa'

            if label_path.exists():
                label_text = label_path.read_text().strip()
                if label_text:
                    for line in label_text.splitlines():
                        parts = line.strip().split()
                        if len(parts) == 5 and parts[0] == '0':
                            class_name = 'varroa'
                            break

            rows.append({'image_path': str(img_path), 'split': split, 'class_name': class_name, 'label': 1 if class_name == 'varroa' else 0})

    df = pd.DataFrame(rows)
    df.to_csv(output_csv_path, index=False)
    return df

VARROA_CSV = f'{CLASSIFICATION_DIR}/varroa_dataset_cls.csv'
HONEYBEE_CSV = f'{CLASSIFICATION_DIR}/honeybee_varroamite_cls.csv'
EXTERNAL_CSV = f'{CLASSIFICATION_DIR}/external_varroa_mites_detector_cls.csv'

df_varroa = create_binary_classification_csv(VARROA_PROCESSED, VARROA_CSV)
df_honeybee = create_binary_classification_csv(HONEYBEE_PROCESSED, HONEYBEE_CSV)
df_external = create_binary_classification_csv(EXTERNAL_PROCESSED, EXTERNAL_CSV)

print(df_varroa.groupby(['split','class_name']).size())
print(df_honeybee.groupby(['split','class_name']).size())
print(df_external.groupby(['split','class_name']).size())

In [ ]:
class VarroaClassificationDataset(Dataset):
    def __init__(self, csv_path, split, transform=None):
        self.df = pd.read_csv(csv_path)
        self.df = self.df[self.df['split'] == split].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        label = int(row['label'])
        if self.transform:
            image = self.transform(image)
        return image, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [ ]:
def build_loaders(csv_path, batch_size=32):
    train_dataset = VarroaClassificationDataset(csv_path, split='train', transform=train_transform)
    valid_dataset = VarroaClassificationDataset(csv_path, split='valid', transform=val_transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, valid_loader

external_valid_dataset = VarroaClassificationDataset(EXTERNAL_CSV, split='valid', transform=val_transform)
external_valid_loader = DataLoader(external_valid_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
def create_model(model_name):
    if model_name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, 2)
    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    else:
        raise ValueError('Unknown model_name')
    return model.to(device)

def train_model(model, train_loader, valid_loader, criterion, optimizer, epochs=10, patience=3):
    best_f1 = -1
    best_state = None
    wait = 0
    history = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        metrics = evaluate_model(model, valid_loader)
        row = {'epoch': epoch + 1, 'train_loss': train_loss / max(1, len(train_loader)), **metrics}
        history.append(row)

        print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {row['train_loss']:.4f} | Acc: {metrics['accuracy']:.4f} | P: {metrics['precision']:.4f} | R: {metrics['recall']:.4f} | F1: {metrics['f1']:.4f}")

        if metrics['f1'] > best_f1:
            best_f1 = metrics['f1']
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print('Early stopping triggered.')
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

def evaluate_model(model, loader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            y_pred.extend(preds.tolist())
            y_true.extend(labels.numpy().tolist())

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred),
    }

In [ ]:
def run_classification_experiment(model_name, csv_path, run_name, epochs=10, patience=3):
    train_loader, valid_loader = build_loaders(csv_path)
    model = create_model(model_name)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    model, history = train_model(model, train_loader, valid_loader, criterion, optimizer, epochs=epochs, patience=patience)

    internal_metrics = evaluate_model(model, valid_loader)
    external_metrics = evaluate_model(model, external_valid_loader)

    run_dir = f'{CLASSIFICATION_RUNS_DIR}/{run_name}'
    os.makedirs(run_dir, exist_ok=True)
    torch.save(model.state_dict(), f'{run_dir}/best_model.pth')
    history.to_csv(f'{run_dir}/history.csv', index=False)

    return model, internal_metrics, external_metrics, history

# ResNet18 - Varroa Dataset
resnet_varroa_model, resnet_varroa_internal, resnet_varroa_external, resnet_varroa_history = run_classification_experiment(
    model_name='resnet18',
    csv_path=VARROA_CSV,
    run_name='resnet18_varroa_dataset'
)

print('Internal:', resnet_varroa_internal)
print('External:', resnet_varroa_external)

# EfficientNet-B0 - Varroa Dataset
efficientnet_varroa_model, efficientnet_varroa_internal, efficientnet_varroa_external, efficientnet_varroa_history = run_classification_experiment(
    model_name='efficientnet_b0',
    csv_path=VARROA_CSV,
    run_name='efficientnet_b0_varroa_dataset'
)

print('Internal:', efficientnet_varroa_internal)
print('External:', efficientnet_varroa_external)

# ResNet18 - HoneyBee VarroaMite
resnet_honeybee_model, resnet_honeybee_internal, resnet_honeybee_external, resnet_honeybee_history = run_classification_experiment(
    model_name='resnet18',
    csv_path=HONEYBEE_CSV,
    run_name='resnet18_honeybee_varroamite'
)

print('Internal:', resnet_honeybee_internal)
print('External:', resnet_honeybee_external)

# EfficientNet-B0 - HoneyBee VarroaMite
efficientnet_honeybee_model, efficientnet_honeybee_internal, efficientnet_honeybee_external, efficientnet_honeybee_history = run_classification_experiment(
    model_name='efficientnet_b0',
    csv_path=HONEYBEE_CSV,
    run_name='efficientnet_b0_honeybee_varroamite'
)

print('Internal:', efficientnet_honeybee_internal)
print('External:', efficientnet_honeybee_external)

# Manual summary table based on recorded experiment outputs.
classification_data = [
    {'Model':'ResNet18','Dataset':'Varroa','Evaluation':'Internal','Accuracy':0.9550,'Precision':0.9071,'Recall':0.8944,'F1-score':0.9007},
    {'Model':'ResNet18','Dataset':'Varroa','Evaluation':'External','Accuracy':0.8820,'Precision':0.9458,'Recall':0.7202,'F1-score':0.8177},
    {'Model':'EfficientNet-B0','Dataset':'Varroa','Evaluation':'Internal','Accuracy':0.9507,'Precision':0.8866,'Recall':0.8991,'F1-score':0.8928},
    {'Model':'EfficientNet-B0','Dataset':'Varroa','Evaluation':'External','Accuracy':0.9005,'Precision':0.9543,'Recall':0.7661,'F1-score':0.8499},
    {'Model':'ResNet18','Dataset':'HoneyBee','Evaluation':'Internal','Accuracy':0.8873,'Precision':0.9440,'Recall':0.8801,'F1-score':0.9109},
    {'Model':'ResNet18','Dataset':'HoneyBee','Evaluation':'External','Accuracy':0.7521,'Precision':0.7233,'Recall':0.5275,'F1-score':0.6101},
    {'Model':'EfficientNet-B0','Dataset':'HoneyBee','Evaluation':'Internal','Accuracy':0.8911,'Precision':0.9074,'Recall':0.9284,'F1-score':0.9178},
    {'Model':'EfficientNet-B0','Dataset':'HoneyBee','Evaluation':'External','Accuracy':0.6661,'Precision':0.5413,'Recall':0.6009,'F1-score':0.5696},
]

df_cls = pd.DataFrame(classification_data)
df_cls['Model_Dataset'] = df_cls['Model'] + ' - ' + df_cls['Dataset']
df_cls.to_csv(f'{GRAPH_DIR}/classification_results_summary.csv', index=False)
df_cls

In [ ]:
def plot_metric(df, metric, title, filename):
    plot_df = df.pivot_table(index='Model_Dataset', columns='Evaluation', values=metric, aggfunc='first')
    ax = plot_df[['Internal', 'External']].plot(kind='bar', figsize=(11, 5))
    plt.title(title)
    plt.xlabel('Model and training dataset')
    plt.ylabel(metric)
    plt.ylim(0, 1)
    plt.xticks(rotation=30, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.legend(title='Evaluation')
    plt.tight_layout()
    save_path = f'{GRAPH_DIR}/{filename}'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved:', save_path)

plot_metric(df_cls, 'F1-score', 'Classification Models Internal and External F1-score Comparison', 'classification_f1_comparison.png')
plot_metric(df_cls, 'Recall', 'Classification Models Internal and External Recall Comparison', 'classification_recall_comparison.png')

external_cls = df_cls[df_cls['Evaluation'] == 'External'].set_index('Model_Dataset')
ax = external_cls['Accuracy'].plot(kind='bar', figsize=(10, 5))
plt.title('Classification Models External Accuracy Comparison')
plt.xlabel('Model and training dataset')
plt.ylabel('External Accuracy')
plt.ylim(0, 1)
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_path = f'{GRAPH_DIR}/classification_external_accuracy.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', save_path)